# 人物スポットライト動画レンダラー（Colab実行用）

スマホから **「ランタイム → すべてのセルを実行」** を1回タップするだけで、
リポジトリのクローンからダミー動画のレンダリングまでを一気に確認できます。

1. セル1：リポジトリ取得・依存関係インストール・ダミー動画の生成
2. セル2：`examples/sample.json` を `/content/out.mp4` にレンダリング
3. セル3：出力動画をノートブック内で再生し、スマホへダウンロード

それより下の「本番用（任意）」セクションは、自分のドライブ内の動画・JSONを使う場合にのみ
手動で書き換えて実行してください（すべてのセルを実行しても、ここでエラーにはなりません）。

## 1. リポジトリの取得・依存関係のインストール・ダミー動画の生成
動作確認用のダミー動画（`examples/dummy_input.mp4`）と
プロジェクトJSON（`examples/sample.json`）をその場で生成します。

In [ ]:
import os

REPO_DIR = "/content/spotlight-reel"

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/digital-twin-creator/spotlight-reel.git {REPO_DIR}
else:
    print("リポジトリは既に取得済みです:", REPO_DIR)

%cd {REPO_DIR}
!pip install -q -r requirements.txt

# ffmpeg が無い場合のみ有効化してください（Colabには通常プリインストール済みです）
# !apt-get -y install ffmpeg

# 動作確認用のダミー動画・プロジェクトJSON・効果音・フォントを生成
!python make_dummy.py

## 2. サンプルJSONをレンダリング
`examples/sample.json`（動画パスはJSON内の指定に従う）を
`/content/out.mp4` にレンダリングします。

In [ ]:
SAMPLE_JSON_PATH = f"{REPO_DIR}/examples/sample.json"
SAMPLE_OUT_PATH = "/content/out.mp4"

!python render.py "{SAMPLE_JSON_PATH}" --out "{SAMPLE_OUT_PATH}"

print("出力ファイル:", SAMPLE_OUT_PATH)
print("サイズ:", os.path.getsize(SAMPLE_OUT_PATH), "bytes")

## 3. 出力動画の再生・スマホへのダウンロード
ノートブック内で動画を再生します。続けて `files.download` が実行され、
スマホのブラウザでダウンロードが自動的に始まります
（ダウンロードの許可を求められた場合は許可してください）。

In [ ]:
from IPython.display import Video, display

display(Video(SAMPLE_OUT_PATH, embed=True, width=360))

from google.colab import files
files.download(SAMPLE_OUT_PATH)

---
## 本番用（任意）
自分で撮影した動画とスマホ用エディタで作成したプロジェクトJSONを使う場合は、
以下のセルを **手動で書き換えてから** 実行してください。
「すべてのセルを実行」でここまで自動的に流れても、
ドライブがマウントされていない・パスが未設定であるだけではエラーになりません
（動画・JSONが見つからない場合はメッセージを表示してスキップします）。

### 3-1. Googleドライブをマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 3-2. 入力・出力パスの指定
スマホ用エディタで作成したプロジェクトJSONと、元動画のドライブ内パスを指定してください。

In [ ]:
# 例: マイドライブ直下の spotlight_reel フォルダに置いた場合
VIDEO_PATH = "/content/drive/MyDrive/spotlight_reel/input.mp4"
JSON_PATH = "/content/drive/MyDrive/spotlight_reel/project.json"
OUT_PATH = "/content/drive/MyDrive/spotlight_reel/output.mp4"

### 3-3. render.py を実行
`VIDEO_PATH` / `JSON_PATH` が実在するファイルを指すように書き換えてから実行してください。
デフォルト値のまま（ファイルが存在しない）実行してもエラー扱いにはせず、
案内を表示してスキップします。

In [ ]:
if not (os.path.isfile(VIDEO_PATH) and os.path.isfile(JSON_PATH)):
    print("VIDEO_PATH または JSON_PATH が見つかりません。上のセルでパスを本番用に書き換えてから再実行してください。")
    print("  VIDEO_PATH:", VIDEO_PATH, "-> 存在:", os.path.isfile(VIDEO_PATH))
    print("  JSON_PATH :", JSON_PATH, "-> 存在:", os.path.isfile(JSON_PATH))
else:
    os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
    cmd = f'python render.py "{JSON_PATH}" --video "{VIDEO_PATH}" --out "{OUT_PATH}"'
    print(cmd)
    !{cmd}

### 3-4. 本番出力の確認
出力はドライブ内の `OUT_PATH` に書き出されています。

In [ ]:
if os.path.isfile(OUT_PATH):
    print("出力ファイル:", OUT_PATH)
    print("サイズ:", os.path.getsize(OUT_PATH), "bytes")
    display(Video(OUT_PATH, embed=False, width=360))
else:
    print("出力ファイルがまだありません（本番用セクションを実行していない場合は正常です）:", OUT_PATH)